In [1]:
from pathlib import Path
import sys
import warnings
import pickle

import numpy as np
import pandas as pd
import plotly.graph_objs as go
import streamlit as st
from prophet import Prophet
from prophet.plot import plot_components_plotly
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

project_dir = next(
    (
        candidate
        for candidate in (
            Path.cwd(),
            Path.cwd() / "aiMLProject",
            Path.cwd().parent / "aiMLProject",
        )
        if (candidate / "db_connect.py").is_file()
    ),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Could not locate aiMLProject/db_connect.py")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from db_connect import fetch_walmart_sales


def fetch_sales_data():
    sales_data = fetch_walmart_sales()
    sales_data["Date"] = pd.to_datetime(
        sales_data["Date"], dayfirst=True, errors="raise"
    )
    return sales_data


def prepare_forecast_data(sales_data):
    return (
        sales_data.groupby("Date", as_index=False)["Weekly_Sales"]
        .sum()
        .rename(columns={"Date": "ds", "Weekly_Sales": "y"})
        .sort_values("ds")
        .reset_index(drop=True)
    )


/Users/adityasharma/mL/aiMLProject/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_sales = fetch_sales_data()
raw_sales.head()


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106


In [3]:
raw_sales.isnull().sum()


Store           0
Date            0
Weekly_Sales    0
Holiday_Flag    0
Temperature     0
Fuel_Price      0
CPI             0
Unemployment    0
dtype: int64

In [4]:
raw_sales.describe()


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,6435.000000,6435,6.435000e+03,6435.000000,6435.000000,6435.000000,6435.000000,6435.000000
mean,23.000000,2011-06-17 00:00:00,1.046965e+06,0.069930,60.663782,3.358607,171.578394,7.999151
min,1.000000,2010-02-05 00:00:00,2.099862e+05,0.000000,-2.060000,2.472000,126.064000,3.879000
25%,12.000000,2010-10-08 00:00:00,5.533501e+05,0.000000,47.460000,2.933000,131.735000,6.891000
50%,23.000000,2011-06-17 00:00:00,9.607460e+05,0.000000,62.670000,3.445000,182.616521,7.874000
75%,34.000000,2012-02-24 00:00:00,1.420159e+06,0.000000,74.940000,3.735000,212.743293,8.622000
max,45.000000,2012-10-26 00:00:00,3.818686e+06,1.000000,100.140000,4.468000,227.232807,14.313000
std,12.988182,NaN,5.643666e+05,0.255049,18.444933,0.459020,39.356712,1.875885


In [5]:
df = prepare_forecast_data(raw_sales)


In [6]:
df.head()


,ds,y
0,2010-02-05,49750740.50
1,2010-02-12,48336677.63
2,2010-02-19,48276993.78
3,2010-02-26,43968571.13
4,2010-03-05,46871470.30


In [7]:
df


,ds,y
0,2010-02-05,49750740.50
1,2010-02-12,48336677.63
2,2010-02-19,48276993.78
3,2010-02-26,43968571.13
4,2010-03-05,46871470.30
...,...,...
138,2012-09-28,43734899.40
139,2012-10-05,47566639.31
140,2012-10-12,46128514.25
141,2012-10-19,45122410.57


In [8]:
df["ds"] = pd.to_datetime(df["ds"], errors="raise")
df


,ds,y
0,2010-02-05,49750740.50
1,2010-02-12,48336677.63
2,2010-02-19,48276993.78
3,2010-02-26,43968571.13
4,2010-03-05,46871470.30
...,...,...
138,2012-09-28,43734899.40
139,2012-10-05,47566639.31
140,2012-10-12,46128514.25
141,2012-10-19,45122410.57


In [9]:
m = Prophet(
    interval_width=0.80,
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
)
model = m.fit(df)


16:43:59 - cmdstanpy - INFO - Chain [1] start processing
16:43:59 - cmdstanpy - INFO - Chain [1] done processing


In [10]:
future = m.make_future_dataframe(periods=100, freq="W-FRI")
forcast = m.predict(future)
forcast.head()


/Users/adityasharma/mL/aiMLProject/.venv/lib/python3.9/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/adityasharma/mL/aiMLProject/.venv/lib/python3.9/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/adityasharma/mL/aiMLProject/.venv/lib/python3.9/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/adityasharma/mL/aiMLProject/.venv/lib/python3.9/site-packages/prophet/forecaster.py:1565: RuntimeWarning: divide by zero encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/adityasharma/mL/aiMLProject/.venv/lib/python3.9/site-packages/prophet/forecaster.py:1565: RuntimeWarning: overflow encountered in matmul
  Xb_a = np.matmul(seasonal_features.values,
/Users/adityasharma/mL/aiMLProject/.venv/lib/py

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2010-02-05,4.713537e+07,4.332107e+07,5.215834e+07,4.713537e+07,4.713537e+07,7.064520e+05,7.064520e+05,7.064520e+05,7.064520e+05,7.064520e+05,7.064520e+05,0.0,0.0,0.0,4.784182e+07
1,2010-02-12,4.713240e+07,4.378643e+07,5.271954e+07,4.713240e+07,4.713240e+07,1.412432e+06,1.412432e+06,1.412432e+06,1.412432e+06,1.412432e+06,1.412432e+06,0.0,0.0,0.0,4.854484e+07
2,2010-02-19,4.712944e+07,4.261018e+07,5.181655e+07,4.712944e+07,4.712944e+07,3.240590e+04,3.240590e+04,3.240590e+04,3.240590e+04,3.240590e+04,3.240590e+04,0.0,0.0,0.0,4.716184e+07
3,2010-02-26,4.712647e+07,4.130815e+07,5.061985e+07,4.712647e+07,4.712647e+07,-1.180089e+06,-1.180089e+06,-1.180089e+06,-1.180089e+06,-1.180089e+06,-1.180089e+06,0.0,0.0,0.0,4.594638e+07
4,2010-03-05,4.712350e+07,4.095253e+07,5.003947e+07,4.712350e+07,4.712350e+07,-1.452818e+06,-1.452818e+06,-1.452818e+06,-1.452818e+06,-1.452818e+06,-1.452818e+06,0.0,0.0,0.0,4.567068e+07


In [11]:
table1 = m.plot(forcast)

In [12]:
table2 = m.plot_components(forcast)

/Users/adityasharma/mL/aiMLProject/.venv/lib/python3.9/site-packages/prophet/forecaster.py:1421: RuntimeWarning: divide by zero encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/adityasharma/mL/aiMLProject/.venv/lib/python3.9/site-packages/prophet/forecaster.py:1421: RuntimeWarning: overflow encountered in matmul
  comp = np.matmul(X, beta_c.transpose())
/Users/adityasharma/mL/aiMLProject/.venv/lib/python3.9/site-packages/prophet/forecaster.py:1421: RuntimeWarning: invalid value encountered in matmul
  comp = np.matmul(X, beta_c.transpose())


In [13]:
with open(project_dir / "prophet_model.pkl", "wb") as f:
    pickle.dump(m, f)


In [14]:
# ---------------- CONFIG ----------------
st.set_page_config(page_title="Forecast Dashboard", layout="wide")
st.title("📊 Forecasting Dashboard")

# ---------------- LOAD DATA & MODEL ----------------
@st.cache_data
def load_data():
    return prepare_forecast_data(fetch_sales_data())

@st.cache_resource
def load_model():
    with open(project_dir / "prophet_model.pkl", "rb") as f:
        return pickle.load(f)

df = load_data()
model = load_model()

# ---------------- SIDEBAR CONTROLS ----------------
st.sidebar.header("⚙️ Settings")
periods = st.sidebar.slider("Forecast horizon (weeks)", 4, 52, 12)
show_components = st.sidebar.checkbox("Show trend/seasonality components", value=False)
show_raw_data = st.sidebar.checkbox("Show raw data table", value=False)

# ---------------- MAIN DASHBOARD ----------------
col1, col2 = st.columns([3, 1])

with col2:
    st.metric("Total Records", len(df))
    st.metric("Date Range", f"{df['ds'].min().date()} → {df['ds'].max().date()}")

with col1:
    st.subheader("Historical Data")
    fig_hist = go.Figure()
    fig_hist.add_trace(go.Scatter(x=df["ds"], y=df["y"], mode="lines", name="Actual"))
    st.plotly_chart(fig_hist, use_container_width=True)

if show_raw_data:
    st.subheader("Raw Data")
    st.dataframe(df)

# ---------------- FORECAST ----------------
st.markdown("---")
st.subheader(f"🔮 {periods}-Week Forecast")

future = model.make_future_dataframe(periods=periods, freq="W-FRI")
forecast = model.predict(future)

fig_forecast = go.Figure()
fig_forecast.add_trace(go.Scatter(x=df["ds"], y=df["y"], name="Actual", line=dict(color="black")))
fig_forecast.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat"], name="Forecast", line=dict(color="blue")))
fig_forecast.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat_upper"], line=dict(width=0), showlegend=False))
fig_forecast.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat_lower"], fill="tonexty", fillcolor="rgba(0,100,255,0.2)", line=dict(width=0), name="Confidence Interval"))

st.plotly_chart(fig_forecast, use_container_width=True)

# Forecast table
st.subheader("Forecast Values")
forecast_display = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(periods)
forecast_display.columns = ["Date", "Predicted", "Lower Bound", "Upper Bound"]
st.dataframe(forecast_display, use_container_width=True)

# Download button
csv = forecast_display.to_csv(index=False).encode("utf-8")
st.download_button("📥 Download Forecast as CSV", csv, "forecast.csv", "text/csv")

# ---------------- COMPONENTS ----------------
if show_components:
    st.markdown("---")
    st.subheader("Trend & Seasonality Components")
    fig_comp = plot_components_plotly(model, forecast)
    st.plotly_chart(fig_comp, use_container_width=True)


2026-10-25 16:43:59.824 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:43:59.825 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:43:59.933 
  command:

    streamlit run /Users/adityasharma/mL/aiMLProject/.venv/lib/python3.9/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-10-25 16:43:59.933 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:43:59.934 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-25 16:43:59.934 No runtime found, using MemoryCacheStorageManager
2026-10-25 16:43:59.934 No runtime found, using MemoryCacheStorageManager
2026-10-25 16:43:59.935 Thread 'MainThread': missing ScriptR